In [3]:
!uv add -qU langchain-teddynote

In [5]:
!uv add -U langchain-openai

Resolved 102 packages in 310ms
Checked 96 packages in 7ms


In [ ]:
!uv add -U langchain-huggingface

In [ ]:
!uv add -U langchain-upstage

In [ ]:
!uv add langchain-ollama

In [9]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

from pathlib import Path

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024,
)

documents = Path("data/01_embedding_documents.csv").read_text(encoding="utf-8")
query  = "설비에서 진동 이상이 발생했어."

document_vectors = embeddings.embed_documents(documents)
query_vector = embeddings.embed_query(query)

print(f"문서 수: {len(document_vectors)}")
print(f"벡터 차원: {len(query_vector)}")
print(f"벡터 일부: {query_vector[:5]}")


문서 수: 1343
벡터 차원: 1024
벡터 일부: [0.005306243896484375, 0.049713134765625, -0.036651611328125, 0.047119140625, 0.02276611328125]


In [11]:
document_vectors = await embeddings.aembed_documents(documents)
query_vector = await embeddings.aembed_query(query)

In [28]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

from pathlib import Path

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024,
)

documents_raw = Path("data/01_embedding_documents.csv").read_text(encoding="utf-8")
query  = "윤활유나 오일이 부족하면 어떻게 해?"

lines = documents_raw.strip().split("\n")[1:]
documents = [line.split(",")[2] for line in lines]
document_vectors = embeddings.embed_documents(documents)
query_vector = embeddings.embed_query(query)

print(f"문서 수: {len(document_vectors)}")
print(f"벡터 차원: {len(query_vector)}")
print(f"벡터 일부: {query_vector[:5]}")

문서 수: 20
벡터 차원: 1024
벡터 일부: [-0.06085205078125, -0.022491455078125, 0.0282745361328125, 0.035125732421875, 0.0026454925537109375]


In [29]:
import numpy as np
def cosine_similarity(a,b):
    a,b = np.array(a), np.array(b)
    return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))

similarities = [cosine_similarity(query_vector, doc_vec) for doc_vec in document_vectors]

top_idx = int(np.argmax(similarities))
print(f"가장 유사한 문서: {documents[top_idx]}")
print(f"유사도: {similarities[top_idx]:.4f}")

가장 유사한 문서: 베어링 윤활유는 6개월 주기로 교체하며 오염도와 점도를 함께 확인합니다.
유사도: 0.3830


In [ ]:
!uv add -U langchain-classic langchain-openai

In [42]:
from time import perf_counter

from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

underlying = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024,
)
store = LocalFileStore("./.cache/embeddings")

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying,
    store,
    namespace="text-embedding-3-small-1024",
    query_embedding_cache=True,
    batch_size=32,
)

texts_raw= Path("data/04_cache_and_duplicate_test.csv").read_text(encoding="utf-8")
lines = texts_raw.strip().split("\n")[1:]
texts = [line.split(",")[2] for line in lines]

start = perf_counter()
first_vectors = cached_embeddings.embed_documents(texts)
first_elapsed = perf_counter() - start

start = perf_counter()
second_vectors = cached_embeddings.embed_documents(texts)
second_elapsed = perf_counter() - start

print(f"첫 실행: {first_elapsed:.3f}초")
print(f"캐시 실행: {second_elapsed:.3f}초")
print(first_vectors == second_vectors)


첫 실행: 0.739초
캐시 실행: 0.040초
True


In [39]:
first_query = cached_embeddings.embed_query("진동 이상")
second_query = cached_embeddings.embed_query("윤활유 이상")

print(first_query == second_query)

False


In [ ]:
!uv add -U langchain-huggingface sentence-transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
load_dotenv()
from pathlib import Path

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32,
    },
)

documents_raw = Path("data/06_multilingual_documents.csv").read_text(encoding="utf-8")
lines = documents_raw.strip().split("\n")[1:]
documents = [line.split(",")[2] for line in lines]

query = "설비 진동에 문제가 있습니다."

document_vectors = embeddings.embed_documents(documents)
query_vector = embeddings.embed_query(query)

scores = [
    sum(query_value * document_value for query_value, document_value in zip(
        query_vector,
        document_vector,
    ))
    for document_vector in document_vectors
]

for text, score in sorted(
    zip(documents, scores),
    key=lambda item: item[1],
    reverse=True,
):
    print(f"{score:.4f} | {text}")

In [1]:
from langchain_ollama import OllamaEmbeddings
from dotenv import load_dotenv
load_dotenv()
from pathlib import Path

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434",
)

documents_raw = Path("data/05_clustering_documents.csv").read_text(encoding="utf-8")
lines = documents_raw.strip().split("\n")[1:]
documents = [line.split(",")[2] for line in lines]

query = "윤활 점검 방법"

document_vectors = embeddings.embed_documents(documents)
query_vector = embeddings.embed_query(query)

print(f"문서 벡터 수: {len(document_vectors)}")
print(f"벡터 차원: {len(query_vector)}")

문서 벡터 수: 16
벡터 차원: 768
